# Chooser on the competitive shootout battery

The competitive shootout is a set of frozen data panels, each scored against the
losses that recorded competitor libraries reached on the same panel. The chooser is
a routine that picks the model capacity for a panel: a base additive fit, a version
with a tree term, or a version with tensor terms. This page runs the chooser on the frozen panels and compares its excess loss with the hand-set hybrid that the shootout recorded.

The excess loss of a method on a panel is its loss relative to the best recorded
competitor on that panel, as a percentage. Zero means the method matched the best
competitor. Negative values mean the method beat it.

The probe (the script that runs the chooser on the panels), its deviations from the recorded study, and every per-seed row are in
`docs/benchmarks/chooser_shootout/results.json`.

(what-changed-from-the-recorded-study)=
## Changes from the recorded study

Two deviations are recorded in the artifact:

- The chooser selects the capacity with five internal folds on the development rows.
  The recorded hybrid tuned its settings on a train and validation split instead.
- The chooser's tree term covers numeric columns only. The recorded hybrid split on
  every column, including categorical ones.

Everything else follows the frozen study: the same payloads, the same base formula (additive smooth terms with four degrees of freedom each), the same refit convention, and the same loss, which is mean squared error on the continuous panels and log loss on the binary panels.

In [1]:
import json
from pathlib import Path

import polars as pl
from great_tables import GT

# This page runs from docs/book/benchmarks; the probe's results sit beside the study.
RESULTS = Path("evidence/chooser_shootout/results.json")
artifact = json.loads(RESULTS.read_text())
rows = pl.DataFrame(artifact["rows"])

print("deviations:")
for item in artifact["deviations"]:
    print(" -", item)
print(f"rows: {rows.height}; statuses: {rows.group_by('status').len().sort('status')['status'].to_list()}")

deviations:
 - select_capacity chooses the form with 5 internal folds on the development rows; the study's arms tuned settings on the train/val split
 - the chooser's tree rung covers numeric columns only; the study's hybrid tree split on every column including categoricals
rows: 45; statuses: ['error', 'ok']


One row failed. `bike_future`, seed 1, returned an error. The panel's summary below
uses its remaining two seeds, and the failure is listed in the failure table.
The chooser's error for that seed is recorded in the artifact.

## Excess loss by panel

For each panel, the table averages the excess loss over the successful seeds. The
bottom row takes the median across panels. Bold marks the lower value in each row.
Lower is better.

In [2]:
# Failures stay out of the means but appear in their own table.
failed = rows.filter(pl.col("status") != "ok").select(["task", "seed", "status"])
ok = rows.filter(pl.col("status") == "ok")

panel = (
    ok.group_by("task")
    .agg(
        pl.col("hybrid_excess_pct").mean().alias("Tuned hybrid"),
        pl.col("chooser_excess_pct").mean().alias("Chooser"),
        pl.col("selected_rung").unique().sort().alias("Rung"),
        pl.len().alias("Seeds"),
    )
    .sort("task")
)
panel = panel.with_columns(pl.col("Rung").list.join(", ").alias("Rung"))
print(f"failed rows: {failed.height}")

failed rows: 1


In [3]:
def bold_lower(hybrid, chooser):
    """Format the two excess values, bolding the lower one."""
    if chooser < hybrid:
        return f"{hybrid:+.1f}%", f"**{chooser:+.1f}%**"
    if hybrid < chooser:
        return f"**{hybrid:+.1f}%**", f"{chooser:+.1f}%"
    return f"**{hybrid:+.1f}%**", f"**{chooser:+.1f}%**"


display_rows = []
for r in panel.iter_rows(named=True):
    hybrid_text, chooser_text = bold_lower(r["Tuned hybrid"], r["Chooser"])
    display_rows.append(
        {
            "Panel": r["task"],
            "Tuned hybrid": hybrid_text,
            "Chooser": chooser_text,
            "Chooser's rung": r["Rung"],
            "Seeds": r["Seeds"],
        }
    )

median_hybrid = panel["Tuned hybrid"].median()
median_chooser = panel["Chooser"].median()
hybrid_text, chooser_text = bold_lower(median_hybrid, median_chooser)
display_rows.append({"Panel": "Median across panels", "Tuned hybrid": hybrid_text, "Chooser": chooser_text, "Chooser's rung": "", "Seeds": None})

excess_table = pl.DataFrame(display_rows)
(
    GT(excess_table)
    .tab_header(
        title="Excess loss against the best recorded competitor, by panel",
        subtitle="Percent above the competitor's loss, averaged over seeds; lower is better; 0% matches the best competitor",
    )
    .fmt_markdown(columns=["Tuned hybrid", "Chooser"])
    .tab_stubhead(label="Panel")
    .tab_source_note(
        "Bold: the lower excess in the row. Averages use successful seeds only. Rung: the form the chooser selected, "
        "base is the additive fit, +tree adds a tree term, +tensors adds tensor terms. Source: rows in results.json."
    )
)

GT(_tbl_data=shape: (14, 5)
┌──────────────────────┬──────────────┬────────────┬─────────────────┬───────┐
│ Panel                ┆ Tuned hybrid ┆ Chooser    ┆ Chooser's rung  ┆ Seeds │
│ ---                  ┆ ---          ┆ ---        ┆ ---             ┆ ---   │
│ str                  ┆ str          ┆ str        ┆ str             ┆ i64   │
╞══════════════════════╪══════════════╪════════════╪═════════════════╪═══════╡
│ bike_future          ┆ **+70.0%**   ┆ +241.6%    ┆ +tree           ┆ 2     │
│ bike_iid             ┆ **+11.3%**   ┆ +294.7%    ┆ base            ┆ 3     │
│ cancer               ┆ +104.0%      ┆ **+7.6%**  ┆ base            ┆ 3     │
│ concrete             ┆ +19.6%       ┆ **+12.7%** ┆ +tree           ┆ 3     │
│ correlated_proxies   ┆ +2.7%        ┆ **-3.5%**  ┆ base            ┆ 5     │
│ …                    ┆ …            ┆ …          ┆ …               ┆ …     │
│ joint_boundary       ┆ **+65.5%**   ┆ +249.1%    ┆ +tensors, +tree ┆ 5     │
│ protein              ┆ **+8.9%**    ┆ +17.1%     ┆ +tree           ┆ 3     │
│ smooth_sparse        ┆ +5.6%        ┆ **-0.9%**  ┆ base            ┆ 5     │
│ superconductivity    ┆ **+7.7%**    ┆ +10.6%     ┆ +tree           ┆ 3     │
│ Median across panels ┆ **+7.9%**    ┆ +12.7%     ┆                 ┆ null  │
└──────────────────────┴──────────────┴────────────┴─────────────────┴───────┘, _body=<great_tables._gt_data.Body object at 0x10d493230>, _boxhead=Boxhead([ColInfo(var='Panel', type=<ColInfoTypeEnum.default: 1>, column_label='Panel', column_align='left', column_width=None), ColInfo(var='Tuned hybrid', type=<ColInfoTypeEnum.default: 1>, column_label='Tuned hybrid', column_align='left', column_width=None), ColInfo(var='Chooser', type=<ColInfoTypeEnum.default: 1>, column_label='Chooser', column_align='left', column_width=None), ColInfo(var="Chooser's rung", type=<ColInfoTypeEnum.default: 1>, column_label="Chooser's rung", column_align='left', column_width=None), ColInfo(var='Seeds', type=<ColInfoTypeEnum.default: 1>, column_label='Seeds', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x10daebf20>, _spanners=Spanners([]), _heading=Heading(title='Excess loss against the best recorded competitor, by panel', subtitle="Percent above the competitor's loss, averaged over seeds; lower is better; 0% matches the best competitor", preheader=None), _stubhead='Panel', _summary_rows=<great_tables._gt_data.SummaryRows object at 0x10cfd9a00>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x10ebd5310>, _source_notes=['Bold: the lower excess in the row. Averages use successful seeds only. Rung: the form the chooser selected, base is the additive fit, +tree adds a tree term, +tensors adds tensor terms. Source: rows in results.json.'], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x10d8e3d70>, _formats=[<great_tables._gt_data.FormatInfo object at 0x10e279070>], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', typ

The median across panels is 12.7% for the chooser and 7.9% for the tuned hybrid. The chooser reduces excess loss on cancer, smooth sparse and correlated proxies.
It increases excess loss on the bike panels, joint boundary and diamonds.

## Failure table

The failed row is listed here with its status. It is kept out of every mean above.

In [4]:
(
    GT(failed)
    .tab_header(title="Failed chooser runs", subtitle="Rows excluded from the means on the previous table")
    .tab_stubhead(label="Panel")
    .tab_source_note("Status as recorded by the probe. The three causes filed against the chooser are discussed in the findings.")
)

GT(_tbl_data=shape: (1, 3)
┌─────────────┬──────┬────────┐
│ task        ┆ seed ┆ status │
│ ---         ┆ ---  ┆ ---    │
│ str         ┆ i64  ┆ str    │
╞═════════════╪══════╪════════╡
│ bike_future ┆ 1    ┆ error  │
└─────────────┴──────┴────────┘, _body=<great_tables._gt_data.Body object at 0x10e413770>, _boxhead=Boxhead([ColInfo(var='task', type=<ColInfoTypeEnum.default: 1>, column_label='task', column_align='left', column_width=None), ColInfo(var='seed', type=<ColInfoTypeEnum.default: 1>, column_label='seed', column_align='right', column_width=None), ColInfo(var='status', type=<ColInfoTypeEnum.default: 1>, column_label='status', column_align='left', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x10ec6a810>, _spanners=Spanners([]), _heading=Heading(title='Failed chooser runs', subtitle='Rows excluded from the means on the previous table', preheader=None), _stubhead='Panel', _summary_rows=<great_tables._gt_data.SummaryRows object at 0x10ec6bf80>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x10e60eb70>, _source_notes=['Status as recorded by the probe. The three causes filed against the chooser are discussed in the findings.'], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x10ec6b560>, _formats=[], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_right_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_right_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_right_color=OptionsInfo(scss=True, category='table', type='value', value='#D3D3D3'), table_border_bottom_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_bottom_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_bottom_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_bottom_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_left_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_left_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_left_color=OptionsInfo(s

## Findings

- **Cancer panel.** The tuned hybrid is 104% above the best competitor, where its
  always-on joint tree overfits. The chooser is about 8% above. On smooth sparse and
  correlated proxies, it is below the best competitor.
- **It loses ground on the bike panels and the joint boundary.** On bike_iid and
  bike_future the chooser is far above the competitor. The bike panels have a
  categorical hour-by-day interaction, and the chooser's tree term reads numeric
  columns only. The gap is that the categorical tree term has no support for categorical columns.
- **The median across panels favours the tuned hybrid.** The chooser is worse on the
  median. The shootout's own hybrid was tuned on the same panels, so the comparison
  is between a tuned hybrid and an automatic choice.
- **The PM2.5 forecast gives a different result.** In the PM2.5 forecast run (its README in the repository), the chooser kept the analyst's base formula, with a held-out RMSE of 0.6604 on the log1p scale, against 0.6615 for the shootout's hybrid and 0.6213 for LightGBM. That run is not in `results.json`, so this page does not recompute it.

## Limits

- **One shared machine.** The losses are accuracy measurements, and load does not
  change them. The timings in the artifact are descriptive.
- **Three causes of the remaining gap are known.** One cause is the categorical tree term. A second is a crash when building a tensor term (a surface over two inputs) for a pair of inputs the chooser selected, where one input takes only two values (bike_future seed 1). The third is a step budget shared across the candidate forms, which undertrains the richer forms.
- **Few seeds per panel.** Nine panels have three seeds and three have five.
  bike_future has two successful seeds. The differences between seeds are not
  reported as intervals.
- **Recorded competitors.** The excess is measured against losses recorded for the
  frozen study. It is not a fresh comparison against the competitor libraries.

## Reproduction

From the repository root, the probe is `benches/probe_chooser_shootout.py`. Its
inputs are the frozen payloads under `benches/.cache/competitive_shootout`, which are
not part of this page.